In [ ]:
import os
import ast
import cv2
import numpy as np
import pandas as pd
import plotly.express as px

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from pattern_recognition.data.loadTraces import LoadTraces
from pattern_recognition.utils.parameters_dics import dbTableNames
from dash_viewer.classification.load_prepare_data import load_queries_table_db
from pattern_recognition.data_preprocessing.speqle.concatenate_traces import CurveConcatenator

from sklearn.decomposition import PCA

# Phillips

In [ ]:
phi_trace_loader = LoadTraces('BL', 'phi_segmentalStrainInterp_T', None, isComponents=None)
phi_DF = phi_trace_loader.tracesDF
phi_DF = phi_DF[~phi_DF['curve_segment'].str.lower().str.contains('apex', na=False)]

# GE

In [ ]:
ste_trace_loader = LoadTraces('BL', 'st_segmentalStrainInterp_T', None, isComponents=None)
ste_DF = ste_trace_loader.tracesDF

## Concatenate the two DataFrames

In [ ]:
contact_st_df = pd.concat([ste_DF, phi_DF], ignore_index=True)
tracesDF = contact_st_df.groupby('study_nr').filter(lambda x: len(x) == 18)

In [ ]:
contact_st_df.shape[0]

In [ ]:
# print(contact_st_df['strain_trace'].head(1).tolist())

In [ ]:
tracesDF['cardiac_view'] = tracesDF['cardiac_view'].str.lower()
tracesDF['segment_wall'] = tracesDF['segment_wall'].str.lower()
dropna_tracestracesDF = tracesDF.dropna(subset=['strain_trace'])
dropna_tracestracesDF = dropna_tracestracesDF[~dropna_tracestracesDF['strain_trace'].apply(lambda x: np.all(np.array(x) == 0))]
tracestracesDF = dropna_tracestracesDF.sort_values(by=['study_nr', 'id_position'])
study_nr = np.sort(tracestracesDF['study_nr'].unique())

In [ ]:
# tracesDF = contact_st_df.groupby('study_nr').filter(lambda x: len(x) == 18)
# tracesDF.shape[0]

In [ ]:
study_nr_imputed = tracesDF.loc[tracesDF['isImputed'] == True, 'study_nr'].unique()
df_isImputed_1 = tracesDF.loc[tracesDF['study_nr'].isin(study_nr_imputed)]
df_isImputed_0 = tracesDF.loc[~tracesDF['study_nr'].isin(study_nr_imputed)]

In [ ]:
concatenated_strain = CurveConcatenator(df_isImputed_1, isReconstructed=True)
concat_traces = concatenated_strain.concatenate_traces_par()


In [ ]:
# study_counts = df_isImputed_0.groupby('study_nr').size()
# print("Number of studies traces original:", len(study_counts[study_counts == 18 ]))

In [ ]:
# concatenated_strain = CurveConcatenator(df_isImputed_1, isReconstructed=True)
# concat_tracesDF_1 = concatenated_strain.concatenate_traces_par()
# concatenated_strain = CurveConcatenator(df_isImputed_0, isReconstructed=True)
# concat_tracesDF_0 = concatenated_strain.concatenate_traces_par()

## Compute mean across rows on train data

In [ ]:
stacked_traces = np.stack(concat_tracesDF_1['concatenated_trace'].values)
mean_trace = np.mean(stacked_traces, axis=0)
print(mean_trace)
len(mean_trace)

In [ ]:
segment_names = concat_tracesDF_1['concat_curve_segm'][0].tolist()

###  Plot the PCA components contribution

In [ ]:
def get_cumulative_component_sum(pca_components_list, component_idx):
    cumulative = np.sum(pca_components_list[:component_idx + 1], axis=0)
    return cumulative


In [ ]:
def generate_modified_trace(mean_trace, cumulative_component, mode='add'):
    if mode == 'add':
        return mean_trace + cumulative_component
    elif mode == 'subtract':
        return mean_trace - cumulative_component
    else:
        raise ValueError("mode must be 'add' or 'subtract'")

# Computing the PCA components contribution

In [ ]:
n_components = 59
pca = PCA(n_components=n_components)
X_transformed = pca.fit_transform(stacked_traces)


In [ ]:
components_ = pca.components_
explained_variance = pca.explained_variance_ratio_
mean_vector = pca.mean_

In [ ]:
X_test = np.stack(concat_tracesDF_0['concatenated_trace'].values)
mean_test_trace = np.mean(X_test, axis=0)

# Plot reconstructed

In [ ]:
def plot_segmental_pca_reconstruction(pca, mean_test_trace, segment_names, max_components=20):
    n_segments = len(segment_names)
    segment_len = len(mean_test_trace) // n_segments

    mean_centered = mean_test_trace - pca.mean_
    mean_projection = mean_centered @ pca.components_.T
    cumulative_sum = np.zeros_like(mean_centered)

    fig = go.Figure()
    color_palette = px.colors.qualitative.Set3

    for step in range(max_components):
        cumulative_sum += mean_projection[step] * pca.components_[step]
        reconstruction = cumulative_sum + pca.mean_

        for seg_idx, seg_name in enumerate(segment_names):
            start = seg_idx * segment_len
            end = (seg_idx + 1) * segment_len
            seg_trace = reconstruction[start:end]

            fig.add_trace(go.Scatter(
                y=seg_trace,
                mode='lines',
                name=f'{seg_name} (PCs {step+1})',
                line=dict(width=2, color=color_palette[seg_idx % len(color_palette)]),
                visible=(step == 0)
            ))

    for seg_idx, seg_name in enumerate(segment_names):
        start = seg_idx * segment_len
        end = (seg_idx + 1) * segment_len
        seg_trace = mean_test_trace[start:end]

        fig.add_trace(go.Scatter(
            y=seg_trace,
            mode='lines',
            name=f'{seg_name} (Original)',
            line=dict(
                width=2,
                dash='dot',
                color=color_palette[seg_idx % len(color_palette)]
            ),
            visible=True
        ))


    steps = []
    total_traces = max_components * n_segments
    for step in range(max_components):
        visibility = [False] * total_traces + [True] * n_segments
        for seg_idx in range(n_segments):
            visibility[step * n_segments + seg_idx] = True

        steps.append(dict(
            method="update",
            label=f"PCs {step+1}",
            args=[{"visible": visibility},
                  {"title": f"Segmental Reconstruction with PCs {step+1}"}]
        ))

    fig.update_layout(
        sliders=[dict(
            active=0,
            currentvalue={"prefix": "Components: "},
            pad={"t": 50},
            steps=steps
        )],
        title="Segment-wise PCA Reconstruction of Mean Strain (Dotted Original)",
        xaxis_title="Timepoints (per segment)",
        yaxis_title="Strain",
        template="plotly_white",
        height=750
    )

    fig.show()


In [ ]:
# plot_segmental_pca_reconstruction(
#     pca=pca,
#     mean_test_trace=mean_test_trace,
#     segment_names=segment_names,
#     max_components=20
# )

In [ ]:
def get_segmental_reconstruction_trace(step, cumulative, mean_centered, pca, mode, seg_name, start, end, output_dir):

    if mode == "add":
        reconstructed = cumulative + pca.mean_
        seg_trace = reconstructed[start:end]
        label = f"Adding PC {step + 1}"
        color = "blue"
        title = f"{seg_name} — Adding PC {step + 1}"
        filename = os.path.join(output_dir, f"{seg_name}_add_{step:03d}.png")
    else:
        reconstructed = pca.mean_ + (mean_centered - cumulative)
        seg_trace = reconstructed[start:end]
        label = f"Subtracting PC {step + 1}"
        color = "red"
        title = f"{seg_name} — Subtracting PC {step + 1}"
        filename = os.path.join(output_dir, f"{seg_name}_sub_{step:03d}.png")

    return seg_trace, label, color, title, filename

In [ ]:
def plot_segmental_trace(seg_trace, original_seg, label, color, title, filename):

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        y=seg_trace,
        mode='lines',
        name=label,
        line=dict(color=color, width=3)
    ))

    fig.add_trace(go.Scatter(
        y=original_seg,
        mode='lines',
        name='Original',
        line=dict(color='black', width=2, dash='dot')
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Timepoints",
        yaxis_title="Strain",
        template="plotly_white",
        height=500,
        width=700,
        legend=dict(orientation="h", x=0, y=-0.2)
    )
    fig.write_image(filename)

In [ ]:
def create_segmental_reconstruction_videos(pca, mean_test_trace, segment_names,
                                            output_dir="segment_videos", fps=3,
                                            max_components=59, mode="add"):

    assert mode in ["add", "subtract"], "mode must be 'add' or 'subtract'"
    os.makedirs(output_dir, exist_ok=True)

    n_segments = len(segment_names)
    segment_len = len(mean_test_trace) // n_segments
    mean_centered = mean_test_trace - pca.mean_
    projection = mean_centered @ pca.components_.T

    for seg_idx, seg_name in enumerate(segment_names):
        print(f"🎥 Processing segment: {seg_name} ({seg_idx + 1}/{n_segments})")

        start = seg_idx * segment_len
        end = (seg_idx + 1) * segment_len
        original_seg = mean_test_trace[start:end]

        cumulative = np.zeros_like(mean_centered)
        frame_paths = []

        for step in range(max_components):
            cumulative += projection[step] * pca.components_[step]

            seg_trace, label, color, title, filename = get_segmental_reconstruction_trace(
                step, cumulative, mean_centered, pca, mode, seg_name, start, end, output_dir
            )

            plot_segmental_trace(seg_trace, original_seg, label, color, title, filename)
            frame_paths.append(filename)

        # Save video
        frame_shape = cv2.imread(frame_paths[0]).shape[1::-1]
        video_path = os.path.join(output_dir, f"{seg_name}_{mode}.mp4")
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(video_path, fourcc, fps, frame_shape)

        for path in frame_paths:
            writer.write(cv2.imread(path))
        writer.release()

        # Clean up
        for path in frame_paths:
            os.remove(path)

    print(f"✅ All '{mode}' videos saved in: {output_dir}")

In [ ]:
# create_segmental_reconstruction_videos(
#     pca=pca,
#     mean_test_trace=mean_test_trace,
#     segment_names=segment_names,
#     output_dir="segmental_add_videos",
#     fps=2,
#     max_components=20,
#     mode="add"
# )
#
# create_segmental_reconstruction_videos(
#     pca=pca,
#     mean_test_trace=mean_test_trace,
#     segment_names=segment_names,
#     output_dir="segmental_sub_videos",
#     fps=2,
#     max_components=20,
#     mode="subtract"
# )


In [ ]:

def create_segmental_add_subtract_comparison_videos(pca, mean_test_trace, segment_names,
                                                     output_dir="segment_videos", fps=3,
                                                     max_components=59):

    os.makedirs(output_dir, exist_ok=True)

    n_segments = len(segment_names)
    segment_len = len(mean_test_trace) // n_segments
    mean_centered = mean_test_trace - pca.mean_
    projection = mean_centered @ pca.components_.T

    for seg_idx, seg_name in enumerate(segment_names):
        print(f"🎥 Creating comparison video for segment: {seg_name} ({seg_idx + 1}/{n_segments})")

        start = seg_idx * segment_len
        end = (seg_idx + 1) * segment_len
        original_seg = mean_test_trace[start:end]

        cumulative_add = np.zeros_like(mean_centered)
        cumulative_sub = np.zeros_like(mean_centered)

        image_paths = []

        for step in range(max_components):
            # Additive reconstruction
            cumulative_add += projection[step] * pca.components_[step]
            recon_add = cumulative_add + pca.mean_
            seg_add = recon_add[start:end]

            # Subtractive reconstruction
            cumulative_sub += projection[step] * pca.components_[step]
            recon_sub = pca.mean_ + (mean_centered - cumulative_sub)
            seg_sub = recon_sub[start:end]

            # Plot
            fig, ax = plt.subplots(figsize=(6, 4))
            ax.plot(seg_add, label=f'Add PC {step+1}', color='blue', linewidth=2)
            ax.plot(seg_sub, label=f'Subtract PC {step+1}', color='red', linewidth=2)
            ax.plot(original_seg, label='Original', linestyle='dotted', color='black', linewidth=2)

            ax.set_title(f"{seg_name} — PCA Component {step+1}")
            ax.set_xlabel("Timepoints")
            ax.set_ylabel("Strain")
            ax.grid(True)
            ax.legend(loc='lower left', fontsize=8)
            fig.tight_layout()

            frame_path = f"{output_dir}/{seg_name}_frame_{step:03d}.png"
            plt.savefig(frame_path)
            plt.close()
            image_paths.append(frame_path)

        # Write video
        video_path = os.path.join(output_dir, f"{seg_name}_compare.mp4")
        first_img = cv2.imread(image_paths[0])
        height, width, _ = first_img.shape
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(video_path, fourcc, fps, (width, height))

        for path in image_paths:
            writer.write(cv2.imread(path))
        writer.release()

        # Cleanup
        for path in image_paths:
            os.remove(path)

    print(f"✅ All combined videos saved in: {output_dir}")


In [ ]:
# create_segmental_add_subtract_comparison_videos(
#     pca=pca,
#     mean_test_trace=mean_test_trace,
#     segment_names=segment_names,
#     output_dir="segmental_combined_videos",
#     fps=2,
#     max_components=59
# )


# Concatenated Mean Strain

In [ ]:
def create_global_trace_video(pca, mean_test_trace, output_dir="global_trace_video", fps=2, max_components=59):

    os.makedirs(output_dir, exist_ok=True)
    mean_centered = mean_test_trace - pca.mean_
    projection = mean_centered @ pca.components_.T

    modes = ["add", "subtract"]

    for mode in modes:
        cumulative = np.zeros_like(mean_centered)
        frame_paths = []

        for step in range(max_components):
            cumulative += projection[step] * pca.components_[step]

            if mode == "add":
                reconstruction = cumulative + pca.mean_
                label = f"Adding PC {step + 1}"
                color = "#0072B2"  # colorblind-safe blue
            else:
                reconstruction = pca.mean_ + (mean_centered - cumulative)
                label = f"Subtracting PC {step + 1}"
                color = "#D55E00"  # colorblind-safe red

            # Plot
            fig, ax = plt.subplots(figsize=(10, 4))
            ax.plot(mean_test_trace, label="Original", color="black", linestyle="solid", linewidth=2)
            ax.plot(reconstruction, label=label, color=color, linestyle="dotted", linewidth=2)
            ax.set_title(f"Global Mean Trace — {label}")
            ax.set_xlabel("Timepoints")
            ax.set_ylabel("Strain")
            ax.grid(True)
            ax.legend(loc='lower right', fontsize=8)
            fig.tight_layout()

            frame_path = os.path.join(output_dir, f"{mode}_frame_{step:03d}.png")
            plt.savefig(frame_path)
            plt.close()
            frame_paths.append(frame_path)

        # Save video
        video_path = os.path.join(output_dir, f"{mode}_global_trace.mp4")
        first_img = cv2.imread(frame_paths[0])
        height, width, _ = first_img.shape
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(video_path, fourcc, fps, (width, height))

        for path in frame_paths:
            writer.write(cv2.imread(path))
        writer.release()

        # Clean up frames
        for path in frame_paths:
            os.remove(path)

    print(f"✅ Global reconstruction videos saved in: {output_dir}")


In [ ]:
# create_global_trace_video(pca, mean_test_trace)

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

def create_global_pca_comparison_video(pca, mean_test_trace, output_dir="global_video", fps=2, max_components=59):

    os.makedirs(output_dir, exist_ok=True)

    mean_centered = mean_test_trace - pca.mean_
    projection = mean_centered @ pca.components_.T

    cumulative_add = np.zeros_like(mean_centered)
    cumulative_sub = np.zeros_like(mean_centered)

    frame_paths = []

    for step in range(max_components):
        cumulative_add += projection[step] * pca.components_[step]
        recon_add = cumulative_add + pca.mean_

        cumulative_sub += projection[step] * pca.components_[step]
        recon_sub = pca.mean_ + (mean_centered - cumulative_sub)

        # Plot
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(mean_test_trace, label="Original Mean", color="black", linestyle="solid", linewidth=2)
        ax.plot(recon_add, label=f"Add PC {step+1}", color="#0072B2", linewidth=2)  # Blue (colorblind-safe)
        ax.plot(recon_sub, label=f"Subtract PC {step+1}", color="#D55E00", linewidth=2)  # Red (colorblind-safe)

        ax.set_title(f"Global Mean Trace — PCA Component {step+1}")
        ax.set_xlabel("Timepoints")
        ax.set_ylabel("Strain")
        ax.grid(True)
        ax.legend(loc='lower right', fontsize=8)
        fig.tight_layout()

        frame_path = os.path.join(output_dir, f"global_frame_{step:03d}.png")
        plt.savefig(frame_path)
        plt.close()
        frame_paths.append(frame_path)

    # Write video
    video_path = os.path.join(output_dir, "global_add_subtract_comparison.mp4")
    first_img = cv2.imread(frame_paths[0])
    height, width, _ = first_img.shape
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(video_path, fourcc, fps, (width, height))

    for path in frame_paths:
        writer.write(cv2.imread(path))
    writer.release()

    # Clean up
    for path in frame_paths:
        os.remove(path)

    print(f"🎬 Comparison video saved at: {video_path}")


In [ ]:
# create_global_pca_comparison_video(
#     pca=pca,
#     mean_test_trace=mean_test_trace,
#     output_dir="global_trace_video",
#     fps=2,
#     max_components=59
# )


# Visualize Variations

In [ ]:
def visualize_pc_variation(pca, mean_trace, pc_idx=0, output_dir="pc_videos", fps=2):

    os.makedirs(output_dir, exist_ok=True)
    pc_vector = pca.components_[pc_idx]
    mean_centered = mean_trace - pca.mean_

    scalars = np.concatenate([
        np.linspace(0.0, 1.0, 13),
        np.linspace(1.0, 0.0, 12)[1:],
        np.linspace(0.0, -1.0, 13),
        np.linspace(-1.0, 0.0, 12)[1:]
    ])

    frame_paths = []

    for i, alpha in enumerate(scalars):
        modified_trace = pca.mean_ + alpha * pc_vector
        label = f"α = {alpha:.2f}"

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(mean_trace, label="Mean", color="black", linestyle="solid", linewidth=2)
        ax.plot(modified_trace, label=label, color="dodgerblue", linestyle="dotted", linewidth=2)
        ax.set_title(f"PC{pc_idx+1} Influence — Frame {i+1}/{len(scalars)}")
        ax.set_xlabel("Timepoints")
        ax.set_ylabel("Strain")
        ax.grid(True)
        ax.legend(loc='lower right', fontsize=8)
        fig.tight_layout()

        frame_path = os.path.join(output_dir, f"pc{pc_idx+1}_frame_{i:03d}.png")
        plt.savefig(frame_path)
        plt.close()
        frame_paths.append(frame_path)

    # Save video
    video_path = os.path.join(output_dir, f"pc{pc_idx+1}_influence.mp4")
    first_img = cv2.imread(frame_paths[0])
    height, width, _ = first_img.shape
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(video_path, fourcc, fps, (width, height))

    for path in frame_paths:
        writer.write(cv2.imread(path))
    writer.release()

    # Clean up
    for path in frame_paths:
        os.remove(path)

    print(f"🎥 Video saved: {video_path}")


In [ ]:
pcs_to_plot = [0, 1, 3, 13, 17, 32]

for pc_idx in pcs_to_plot:
    visualize_pc_variation(pca=pca, mean_trace=mean_test_trace, pc_idx=pc_idx)